In [ ]:
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, ParameterSampler
import warnings
warnings.filterwarnings('ignore')

STORE = "CA_2"
OTHER = r"D:\M5 Forecasting"

store_train = pd.read_pickle(OTHER + rf"\Sale-Store\sales-{STORE}.pkl")
# already sorted by Date — confirmed

cols = ['dept_id_enc_mean','item_id_enc_mean','sales','state_id_enc_mean','cat_id_enc_mean',
        'item_id_cat_id_enc_mean','dept_id_item_id_enc_mean','state_dept_id_enc_mean',
        'state_item_id_enc_mean','state_item_dept_enc_mean','store_cat_id_enc_mean',
        'store_dept_id_enc_mean','store_item_id_enc_mean','sales_lag_29','sales_lag_30',
        'sales_lag_31','sales_lag_35','sales_lag_42','sales_lag_58','sellingTrend']
for i in cols:
    if i in store_train.columns:
        store_train[i] = pd.to_numeric(store_train[i])

rmsse_lookup = pd.read_pickle(OTHER + r"\Sale-Store\rmsselookup.pkl")
scale_lookup = rmsse_lookup.get(STORE, {})

def rmsse_from_scale(y_true, y_pred, scale):
    if scale is None or np.isnan(scale) or scale == 0:
        return np.nan
    mse = np.mean((y_true - y_pred) ** 2)
    return np.sqrt(mse / scale)

X = store_train.drop("sales", axis=1)
y = store_train["sales"]

if "item_id" not in X.columns:
    raise ValueError("item_id column needed in X to compute per-item RMSSE — check store_train columns")

tscv = TimeSeriesSplit(n_splits=3)
folds = list(tscv.split(X))

param_distributions = {
    "n_estimators": [250, 500, 750, 1000],
    "learning_rate": [0.01, 0.02, 0.05, 0.1],
    "num_leaves": [30, 50, 70, 100, 150, 250, 200, 300, 450, 350, 500, 1000],
    "max_depth": [5, 15, 25],
    "subsample": [0.6, 0.75, 0.8],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "min_data_in_leaf": [100, 200, 300, 1000],
    "max_bin": [100, 200, 250],
}

param_list = list(ParameterSampler(param_distributions, n_iter=40, random_state=42))

results = []

for trial_idx, params in enumerate(param_list, start=1):
    params_full = dict(params)
    params_full.update({
        "objective": "tweedie",
        "tweedie_variance_power": 1.1,
        "boost_from_average": False,
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    })

    fold_rmse_scores = []
    fold_rmsse_scores = []

    for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_vl, y_vl = X.iloc[valid_idx], y.iloc[valid_idx]

        model = lgb.LGBMRegressor(**params_full)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_tr, y_tr), (X_vl, y_vl)],
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )

        preds = model.predict(X_vl, num_iteration=model.best_iteration_)
        preds = np.clip(preds, 0, None)

        # pooled RMSE (kept for comparison/diagnostics)
        rmse = np.sqrt(np.mean((y_vl.values - preds) ** 2))
        fold_rmse_scores.append(rmse)

        # per-item RMSSE — the metric that actually matches the competition
        fold_df = pd.DataFrame({
            "item_id": X_vl["item_id"].values,
            "y_true": y_vl.values,
            "y_pred": preds,
        })
        item_scores = []
        for item, g in fold_df.groupby("item_id"):
            scale = scale_lookup.get(item, np.nan)
            score = rmsse_from_scale(g["y_true"].to_numpy(), g["y_pred"].to_numpy(), scale)
            if not np.isnan(score):
                item_scores.append(score)
        fold_rmsse_scores.append(np.mean(item_scores) if item_scores else np.nan)

    results.append({
        "params": params_full,
        "mean_rmse": np.mean(fold_rmse_scores),
        "std_rmse": np.std(fold_rmse_scores),
        "mean_rmsse": np.nanmean(fold_rmsse_scores),
        "std_rmsse": np.nanstd(fold_rmsse_scores),
        "fold_rmse_scores": fold_rmse_scores,
        "fold_rmsse_scores": fold_rmsse_scores,
    })

    print(f"Trial {trial_idx}/{len(param_list)}: "
          f"mean_rmse={results[-1]['mean_rmse']:.4f} | "
          f"mean_rmsse={results[-1]['mean_rmsse']:.4f} std_rmsse={results[-1]['std_rmsse']:.4f}")

results_sorted = sorted(results, key=lambda r: r["mean_rmsse"])
for r in results_sorted:
    print(f"mean_rmsse={r['mean_rmsse']:.4f} std_rmsse={r['std_rmsse']:.4f} "
          f"(mean_rmse={r['mean_rmse']:.4f}) params={r['params']}")

best = results_sorted[0]
print("\nBEST PARAMS (by RMSSE):", best["params"])

Trial 1/40: mean_rmse=1.8887 | mean_rmsse=0.6850 std_rmsse=0.0599
Trial 2/40: mean_rmse=1.8900 | mean_rmsse=0.6839 std_rmsse=0.0601
Trial 3/40: mean_rmse=1.8894 | mean_rmsse=0.6841 std_rmsse=0.0604
Trial 4/40: mean_rmse=1.8868 | mean_rmsse=0.6838 std_rmsse=0.0602
Trial 5/40: mean_rmse=1.8900 | mean_rmsse=0.6863 std_rmsse=0.0599
Trial 6/40: mean_rmse=1.8882 | mean_rmsse=0.6857 std_rmsse=0.0600
Trial 7/40: mean_rmse=1.8857 | mean_rmsse=0.6839 std_rmsse=0.0605
Trial 8/40: mean_rmse=1.8929 | mean_rmsse=0.6870 std_rmsse=0.0599
Trial 9/40: mean_rmse=1.8842 | mean_rmsse=0.6834 std_rmsse=0.0598
Trial 10/40: mean_rmse=1.8927 | mean_rmsse=0.6844 std_rmsse=0.0598
Trial 11/40: mean_rmse=1.8856 | mean_rmsse=0.6843 std_rmsse=0.0599
Trial 12/40: mean_rmse=1.8878 | mean_rmsse=0.6841 std_rmsse=0.0596
Trial 13/40: mean_rmse=1.8869 | mean_rmsse=0.6837 std_rmsse=0.0601
Trial 14/40: mean_rmse=1.8883 | mean_rmsse=0.6847 std_rmsse=0.0597
Trial 15/40: mean_rmse=1.8983 | mean_rmsse=0.6873 std_rmsse=0.0603
Tria

In [ ]:
{'subsample': 0.6, 'num_leaves': 300, 'n_estimators': 1000, 'min_data_in_leaf': 100, 'max_depth': 15, 'max_bin': 200, 'learning_rate': 0.01, 'colsample_bytree': 0.7, 'objective': 'tweedie', 'tweedie_variance_power': 1.1, 'boost_from_average': False, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}

In [ ]:
results_sorted = sorted(results, key=lambda r: r["mean_rmse"])
for r in results_sorted:
    print(f"mean_rmsse={r['mean_rmsse']:.4f} std_rmsse={r['std_rmsse']:.4f} "
          f"(mean_rmse={r['mean_rmse']:.4f}) params={r['params']}")

best = results_sorted[0]
print("\nBEST PARAMS (by RMSSE):", best["params"])
lgm_params=  params={'subsample': 0.75, 'num_leaves': 150, 'n_estimators': 500, 'min_data_in_leaf': 200, 'max_depth': 15, 'max_bin': 100, 'learning_rate': 0.01, 'colsample_bytree': 0.7, 'objective': 'tweedie', 'tweedie_variance_power': 1.1, 'boost_from_average': False, 'random_state': 42, 'n_jobs': -1, 'verbose': -1} 
arams={'subsample': 0.6, 'num_leaves': 300, 'n_estimators': 1000, 'min_data_in_leaf': 100, 'max_depth': 15, 'max_bin': 200, 'learning_rate': 0.01, 'colsample_bytree': 0.7, 'objective': 'tweedie', 'tweedie_variance_power': 1.1, 'boost_from_average': False, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}


mean_rmsse=0.6853 std_rmsse=0.0520 (mean_rmse=1.8809) params={'subsample': 0.8, 'num_leaves': 50, 'n_estimators': 500, 'min_data_in_leaf': 200, 'max_depth': 15, 'max_bin': 250, 'learning_rate': 0.1, 'colsample_bytree': 0.7, 'objective': 'tweedie', 'tweedie_variance_power': 1.1, 'boost_from_average': False, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
mean_rmsse=0.6857 std_rmsse=0.0519 (mean_rmse=1.8826) params={'subsample': 0.6, 'num_leaves': 300, 'n_estimators': 1000, 'min_data_in_leaf': 100, 'max_depth': 15, 'max_bin': 200, 'learning_rate': 0.01, 'colsample_bytree': 0.7, 'objective': 'tweedie', 'tweedie_variance_power': 1.1, 'boost_from_average': False, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
mean_rmsse=0.6854 std_rmsse=0.0518 (mean_rmse=1.8831) params={'subsample': 0.75, 'num_leaves': 150, 'n_estimators': 500, 'min_data_in_leaf': 200, 'max_depth': 15, 'max_bin': 100, 'learning_rate': 0.01, 'colsample_bytree': 0.7, 'objective': 'tweedie', 'tweedie_variance_power': 1.1, '

In [ ]:
#non recursive


In [ ]:
import lightgbm as lgm
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, ParameterSampler
import gc
import pickle

STORE = "CA_1"
OTHER = r"D:\M5 Forecasting"

SHIFT_WINDOW_PAIRS = [(s, w) for s in (1, 7) for w in (7, 14, 30, 60)]
DYNAMIC_COLS = [f"rolling_mean_{s}_{w}" for s, w in SHIFT_WINDOW_PAIRS]

sales = pd.read_pickle(OTHER + rf"\Sale-Store\sales-{STORE}.pkl")
sales = sales.drop(columns=[c for c in DYNAMIC_COLS if c in sales.columns])
sales = sales.sort_values("Date").reset_index(drop=True)

numeric_cols = [
    'dept_id_enc_mean', 'dept_id_enc_std',
    'item_id_enc_mean', 'item_id_enc_std',
    'state_item_id_enc_mean', 'state_item_id_enc_std',
    'cat_id_enc_mean', 'cat_id_enc_std',
    'state_dept_id_enc_mean', 'state_dept_id_enc_std',
    'store_dept_id_enc_mean', 'store_dept_id_enc_std',
    'store_cat_id_enc_std',
    'store_item_id_enc_mean', 'store_item_id_enc_std',
    'sales_lag_29', 'sales_lag_30', 'sales_lag_35', 'sales_lag_42',
    'sellingTrend',
]
for col in numeric_cols:
    if col in sales.columns:
        sales[col] = pd.to_numeric(sales[col])
sales["sales"] = pd.to_numeric(sales["sales"])

feature_cols = [c for c in sales.columns if c not in ("sales", "store_id")]

X = sales[feature_cols]
y = sales["sales"]


tscv = TimeSeriesSplit(n_splits=3)
folds = list(tscv.split(X))


param_distributions = {
    "n_estimators":     [300, 500, 750, 1000, 1500],
    "learning_rate":    [0.01, 0.02, 0.05, 0.1],
    "num_leaves":       [50, 70, 100, 150, 250],
    "max_depth":        [10, 15, 25, -1],
    "min_data_in_leaf": [500, 1000, 2000, 4095],
    "subsample":        [0.5, 0.6, 0.75, 0.8],
    "colsample_bytree": [0.5, 0.7, 0.85, 1.0],
    "max_bin":          [100],
}

param_list = list(ParameterSampler(param_distributions, n_iter=30, random_state=42))

results = []

for trial_idx, params in enumerate(param_list, start=1):
    params_full = dict(params)
    params_full.update({
        "objective": "tweedie",
        "tweedie_variance_power": 1.1,
        "boost_from_average": False,
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    })

    fold_scores = []

    for fold_num, (train_idx, valid_idx) in enumerate(folds, start=1):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_vl, y_vl = X.iloc[valid_idx], y.iloc[valid_idx]

        model = lgm.LGBMRegressor(**params_full)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_vl, y_vl)],          
            eval_metric="rmse",
            callbacks=[lgm.early_stopping(30, verbose=False)]
        )

        preds = model.predict(X_vl, num_iteration=model.best_iteration_)
        rmse = np.sqrt(np.mean((y_vl.values - preds) ** 2))
        fold_scores.append(rmse)

        del model
        gc.collect()

    results.append({
        "params": params_full,
        "mean_rmse": float(np.mean(fold_scores)),
        "std_rmse": float(np.std(fold_scores)),
        "fold_scores": fold_scores,
    })

    print(f"Trial {trial_idx}/{len(param_list)}: mean_rmse={results[-1]['mean_rmse']:.4f} std={results[-1]['std_rmse']:.4f}")




results_sorted = sorted(results, key=lambda r: r["mean_rmse"])

print("\n===== TOP 5 =====")
for r in results_sorted[:5]:
    print(f"mean_rmse={r['mean_rmse']:.4f}  std={r['std_rmse']:.4f}  params={r['params']}")

best = results_sorted[0]
print("\nBEST PARAMS:", best["params"])


with open(OTHER + rf"\Results\best_nonrecursive_params_{STORE}.pkl", "wb") as f:
    pickle.dump(best["params"], f)

d:\Python\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\Python\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\Python\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Trial 1/30: mean_rmse=2.7269 std=0.3257


d:\Python\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\Python\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\Python\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Trial 2/30: mean_rmse=2.7154 std=0.3106


d:\Python\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\Python\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\Python\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


KeyboardInterrupt: 